<a href="https://colab.research.google.com/github/daehyun99/BurnFit-AI-developer-assignment/blob/main/%5BDeploy%5DBurnfit-FastAPI-server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. 설정
- `!pip install -q gemma`의 경우, 세션 재시작 필요

In [ ]:
# Gemma 라이브러리 설정
!pip install -q gemma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 52.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.3/122.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.3/479.3 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.4/400.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 18.8 MB/s eta 0

In [ ]:
# 깃허브 클론
!git clone https://github.com/daehyun99/BurnFit-AI-developer-assignment

Cloning into 'BurnFit-AI-developer-assignment'...
remote: Enumerating objects: 229, done.
remote: Counting objects: 100% (229/229), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 229 (delta 87), reused 197 (delta 62), pack-reused 0 (from 0)
Receiving objects: 100% (229/229), 649.73 KiB | 5.37 MiB/s, done.
Resolving deltas: 100% (87/87), done.


In [ ]:
%cd ./BurnFit-AI-developer-assignment/

/content/BurnFit-AI-developer-assignment


In [ ]:
!pip install -q pyngrok

In [ ]:
!pip install -q uvicorn fastapi python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.4 MB/s eta 0:00:00


### 환경변수 설정
1. Gemma 모델을 구글 드라이브에 업로드
2. .env 파일 생성 및 Gemma 모델 경로 설정
3. ngrok 토큰 설정

In [ ]:
# prompt: google drive mount

# Gemma3-1B 모델 로드를 위한 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# .env 파일 생성 및 Gemma3 모델 경로 설정
!echo "GEMMA_MODEL_PATH='/content/drive/MyDrive/05_[공유파일]/Burnfit-model/Gemma3-lora6'" >> .env # 모델 경로 수정 필요!

In [ ]:
!cat .env

GEMMA_MODEL_PATH='/content/drive/MyDrive/05_[공유파일]/Burnfit-model/Gemma3-lora6'


In [ ]:
# ngrok 토큰 입력
!ngrok config add-authtoken [토큰 기입]

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


# 배포
1. 코드 실행 시, 배포까지 시간이 다소 걸립니다. (Gemma 모델을 로드해야해서 1분 이상 걸리는 것 같습니다.)
  - `ERR_NGROK_8012` 오류가 발생하면, 조금 기다리시면 됩니다. (새로고침)
2. 배포 성공하면, ngrok 링크에 `/docs` 추가해서 접속하면 됩니다.

- example
  > 🌍 Public URL: NgrokTunnel: "https://a076-34-170-40-206.ngrok-free.app" -> "http://localhost:8000" 의 경우
  
  > "https://a076-34-170-40-206.ngrok-free.app/docs" 로 접속

In [ ]:
from pyngrok import ngrok
import uvicorn
import threading
import time

def is_server_running():
    import requests
    try:
        response = requests.get("http://127.0.0.1:8000")
        return response.status_code == 200
    except:
        return False

def run():
    uvicorn.run("app.main:app", host="0.0.0.0", port=8000, reload=False)

# FastAPI 서버 실행 (백그라운드)
thread = threading.Thread(target=run)
thread.start()

# FastAPI가 실행될 때까지 대기
time.sleep(60)

# Ngrok을 이용한 터널 생성 (FastAPI 실행 확인 후)
public_url = ngrok.connect(8000)
print(f"🌍 Public URL: {public_url}")


🌍 Public URL: NgrokTunnel: "https://5886-35-223-87-0.ngrok-free.app" -> "http://localhost:8000"


## Command (필요한 경우)

In [ ]:
# 8000번 포트 사용 확인
!lsof -i :8000

In [ ]:
# 8000번 포트 사용 중인 프로세스 삭제
!kill -9 $(lsof -t -i:8000)

kill: usage: kill [-s sigspec | -n signum | -sigspec] pid | jobspec ... or kill -l [sigspec]


In [ ]:
# FastAPI 서버 배포 상태 확인
!curl http://127.0.0.1:8000

INFO:     127.0.0.1:33220 - "GET / HTTP/1.1" 404 Not Found
{"detail":"Not Found"}

In [ ]:
# Ngrok 서버 삭제
ngrok.kill()
